# Perron: High-Throughput Spectral Code Graph Retrieval
### Quickstart & Multi-Instance Offline Benchmark
**Author**: Sultan Haikal ([GitHub: @yvliet](https://github.com/yvliet/perron))  
**License**: Apache-2.0  

This notebook demonstrates turnkey offline execution of **Perron**, a high-throughput spectral code graph diffusion engine for developer agents on consumer hardware.

### Key Capabilities:
1. **Sub-millisecond Spectral Diffusion**: O(K + E) Personalized PageRank over Compressed Sparse Row (CSR) multigraphs.
2. **Query-Directed Hub Suppression**: Resolves scale-free hub explosion without brittle blocklists.
3. **Offline Zero-Copy Ingestion**: Evaluates without network calls or external API keys.


In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import scipy.sparse as sp

# Ensure perron is on Python path
if Path('perron').exists() or Path('../perron').exists():
    sys.path.insert(0, str(Path.cwd()))
    sys.path.insert(0, str(Path.cwd().parent))

from perron.loaders import load_instance_graph
from perron.retriever import CodeGraph, PerronRetriever
from perron.bench import run_benchmark, format_benchmark_report
print('Perron modules loaded successfully.')


## 1. Inspecting a Pre-Indexed Repository Multigraph
We load a pre-compiled SWE-bench Lite instance graph from `data_release/graphs/` and inspect its topological properties.


In [ ]:
graphs_dir = Path('data_release/graphs')
if not graphs_dir.exists():
    graphs_dir = Path('../data_release/graphs')

sample_inst = 'psf__requests-2148'
inst_path = graphs_dir / sample_inst
if not inst_path.exists():
    sample_dirs = sorted([d for d in graphs_dir.iterdir() if d.is_dir()])
    inst_path = sample_dirs[0]

graph = load_instance_graph(inst_path)
print(f'Loaded instance: {graph.instance_id}')
print(f'Nodes: {graph.num_nodes} | Edges: {graph.num_edges}')
print('Top-5 Hub Symbols:')
for name, deg in graph.top_hubs(k=5):
    print(f'  - {name:<35} (in-degree: {deg})')


## 2. Spectral Diffusion Retrieval & Hub Suppression
We execute a natural language bug query over the repository code graph under strict token budgets (K <= 4,096).


In [ ]:
symbols_list = [
    type('Sym', (), {
        'node_id': int(r.id),
        'file_path': str(r.file),
        'qualified_name': str(r.qualified_name),
        'symbol_type': str(r.kind),
        'start_line': int(r.start_line),
        'end_line': int(r.end_line),
        'code': f'# {r.qualified_name} in {r.file}',
        'token_count': 50,
    })()
    for r in graph.nodes_df.itertuples()
]

cg = CodeGraph(
    t_matrix=graph.t_matrix,
    dangling=graph.dangling,
    symbols=symbols_list,
    symbol_index={s.qualified_name: i for i, s in enumerate(symbols_list)},
)

retriever = PerronRetriever(cg, gamma=0.70, beta=0.85)
query_text = 'fix socket error when handling streaming chunked response'
t0 = time.perf_counter()
context = retriever.query(query_text, max_tokens=1024)
elapsed_ms = (time.perf_counter() - t0) * 1000.0
print(f'Diffusion retrieval completed in {elapsed_ms:.2f} ms\n')
print('--- Retrieved Context Preview ---')
print(context[:400] + '...')


## 3. End-to-End Evaluation on 5 Diverse Instances
We run 5 instances across different repositories and print a comparative summary table.


In [ ]:
test_instances = [
    'psf__requests-2148',
    'pallets__flask-4992',
    'mwaskom__seaborn-3190',
    'pytest-dev__pytest-5221',
    'sympy__sympy-13437',
]

print(f"{'Instance ID':<26} | {'Nodes':>7} | {'Edges':>7} | {'One-Step SpMV':>18}")
print('-' * 68)

verified = 0
for iid in test_instances:
    ipath = graphs_dir / iid
    if not ipath.exists():
        raise FileNotFoundError(f'Missing instance graph: {ipath}')
    g = load_instance_graph(ipath)
    t0 = time.perf_counter()
    p = np.zeros(g.num_nodes, dtype=np.float32)
    p[0] = 1.0
    p_next = 0.85 * (g.t_matrix.T @ p) + 0.15 * p
    lat_ms = (time.perf_counter() - t0) * 1000.0
    print(f"{iid:<26} | {g.num_nodes:>7d} | {g.num_edges:>7d} | {lat_ms:>15.2f} ms")
    verified += 1
print('-' * 68)
print(f'{verified}/{len(test_instances)} instances verified in offline execution.')


## 4. Canonical Benchmark Verification
Evaluating the Hub-Gold macro-average metric across the dev-val partition.


In [ ]:
try:
    res = run_benchmark(method='perron', split='dev_val')
    print(format_benchmark_report(res))
except Exception as e:
    print(f'Benchmark runner notice: {e}')
